## Understand and Practice Claude Streams

Basically the idea is to send a series of events(chunks of response) instead of the whole message.

Why it makes the user experience better is because if claude takes for 20-30 seconds, we keep the user engaged by sharing whatever content has been generated so far.

In [ ]:
#install dependent py modules
from anthropic import Anthropic
from dotenv import load_dotenv
from IPython.display import display, Markdown, clear_output

load_dotenv()

In [ ]:
# Params
model = "claude-sonnet-5"

# SDK client
client = Anthropic()

In [ ]:
#helper functions
def add_user_message(messages, text):
    message = { "role": "user", "content": text}
    messages.append(message)

def add_assistant_message(messages, text):
    message = { "role": "assistant", "content": text}
    messages.append(message)

def chat(messages, system_prompt=None):
    params = {
        "model": model,
        "max_tokens": 1024,
        "messages": messages,
    }

    if system_prompt:
        params["system"] = system_prompt

    response = client.messages.create(**params)
    text_blocks = [block.text for block in response.content if block.type == "text"]
    return "\n".join(text_blocks)

# Prettify claude md response
def display_turn(role, text):
    display(Markdown(f"**{role}:**\n\n{text}"))

In [ ]:
messages = []

add_user_message(messages, "Hello Sonnet! - Give me a one line definition of what a llm is? What is the main architecture it uses?")
response_text = chat(messages)

display_turn("Assistant", response_text)

In [ ]:
messages = []

# add_user_message(messages, "Hello Sonnet! - Give me a one line definition of what a llm is? What is the main architecture it uses?")
add_user_message(messages, "Can you do a web search of what is a Money plant?")

streams = client.messages.create(
    model=model,
    max_tokens=1024,
    messages=messages,
    stream=True
)

# kept intentionally as a raw inspection cell -- this is the full,
# unfiltered list of event types the API actually sends during a
# stream (message_start, content_block_delta, message_delta,
# message_stop, etc.), before any of it gets cleaned up into the
# text_stream/final_message pattern used below.
for event in streams:
    print(event.type)
    display_turn("Assistant", event)

In [ ]:
messages = []

add_user_message(messages, "Hello Claude, Can you tell what is a money plant?")

with client.messages.stream(
    model=model,
    max_tokens=1024,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

    final_message = stream.get_final_message()

# clear the raw streaming text now that we have the complete message
clear_output(wait=True)

text_blocks = [block.text for block in final_message.content if block.type == "text"]
response_text = "\n".join(text_blocks)

add_assistant_message(messages, response_text)
display_turn("Assistant", response_text)